In [0]:
dbutils.widgets.text('p_batch_id', '')
v_batch_id = dbutils.widgets.get('p_batch_id')

In [0]:
%run ../common/environmental_config

In [0]:
%run ../common/gold-helper

In [0]:
from pyspark.sql import functions as F

In [0]:
target_table = f"{catalog_name}.{gold_schema}.dim_drivers"

In [0]:
df_drivers = (spark.table(f"{catalog_name}.{silver_schema}.drivers").filter(F.col('batch_id') == v_batch_id))
df_ref_nationality_region = spark.table(f"{catalog_name}.{gold_schema}.ref_nationality_region")

In [0]:
df_dim_drivers = (
    df_drivers.join(df_ref_nationality_region, df_drivers.nationality == df_ref_nationality_region.nationality, "left")
    .select(
        df_drivers.driver_id,
        df_drivers.driver_name,
        df_drivers.date_of_birth,
        df_drivers.nationality,
        df_ref_nationality_region.region.alias('nationality_region'))
)

In [0]:
write_to_gold(
    input_df=df_dim_drivers,
    target_table=target_table,
    merge_condition='t.driver_id = s.driver_id',
    columns_to_update=[
        "driver_name",
        "date_of_birth",
        'nationality',
        'nationality_region'
    ]
)